# Lesson 3 : Language Modelling
__Teaching Machines to predict next word__

*its doing one single things at its core:

__| Given the word so for predict next word.__

The entire field of large language models is an elaborate answer to this one question.


# What is Language model ?

A Language model assigns probablities to sequence of words.

P("cat sat on the mat") = 0.0023 <- reasonable english

P("mat sat on the cat") = 0.0008 <- grammatical but weird

P("sat the on mat the cat") = 0.000001 <- garbage

__more usefully, it answers. "Given everything so far what comes next ?"__

P(next word | "The cat sat on the ___ ")

P("mat") = 0.34

P("floor") = 0.21

P("roof") = 0.08

P("dog")= 0.01

.....

## Why is this enough to build ChatGPT ?

Because if you can predict the next word well enough, you implicitly must understand :

* Grammar (to predict grammatical continuations)

* Facts (to predict factually correct continuations)

* Reasoning ( to predict logically consistent continuations)

* style (to preddict contextually appropriate continuations)

__the "predict next word" objective forces the model to learn evrything about the language__

## Chapter 1 : N-Gram Language models

the simplest possible language model

Count how often word sequences appear in training data. use those counts as probability.

__Unigram model__ - each word independent:

P("cat") = Count("cat")/total words

__Bigram model__ - Probability depends on previous 1 word:

P("sat" | "cat") = Count( "cat sat")/Count("cat")

__trigram Model__ - Probability depends on previous 2 word:

P("sat" | "the cat") = Count("sat")/Count("the cat")



In [ ]:
# N-Gram Language Model from Scratch
from collections import defaultdict, Counter
import random
import math
class NgramLanguageModel:
    def __init__(self,n):
        self.n=n
        self.count=defaultdict(Counter)
        self.vocab=set()

    def train(self,text):
        words=text.lower().split()
        self.vocab.update(words)
        for i in range (len(words)-self.n+1):
            context=tuple(words[i:i+self.n-1])
            target=words[i+self.n-1]
            self.count[context][target]+=1

    def probability(self,word,context):
        context=tuple(context[-self.n-1:])
        total=sum(self.count[context].values())
        if total==0:
            return 0.0
        return (self.count[context][word]+1)/(total+len(self.vocab))
    
    def predict_next(self,context,topn=5):
        context=tuple(context[-self.n+1:])
        scores={}
        for word in self.vocab:
            scores[word]=self.probability(word,context)
        return sorted(scores.items(),key=lambda x: -x[1])[:topn]
    def generate(self,seed,max_words=20):
        words=seed.lower().split()
        for _ in range(max_words):
            predictions = self.predict_next(words)
            # Sample from top predictions (not just argmax — adds variety)
            candidates, probs = zip(*predictions)
            total_probs=sum(probs)
            if total_probs==0:
                # Fallback: If everything is 0, give every word an equal chance
                probs = [1 / len(candidates)] * len(candidates)
            else:
                probs = [p / sum(probs) for p in probs]
            next_word = random.choices(candidates, weights=probs, k=1)[0]
            words.append(next_word)
        return " ".join(words)

    def perplexity(self, text):
        """
        Perplexity = how 'surprised' is the model by this text?
        Lower = better. A perplexity of K means the model is as confused
        as if it had to choose uniformly among K words at each step.
        """
        words   = text.lower().split()
        log_prob = 0
        count    = 0
        for i in range(self.n - 1, len(words)):
            context = words[i - (self.n-1) : i]
            word    = words[i]
            p       = self.probability(word, context)
            log_prob += math.log(p + 1e-10)
            count    += 1
        return math.exp(-log_prob / count)
    




text = """the king rules the kingdom with wisdom the queen advises the king
the prince will one day rule the kingdom the princess studies ancient wisdom
the knight protects the kingdom from enemies the wizard advises with magic
the king and queen rule together with justice for all people in the kingdom
the ancient kingdom has many rules that all people must follow
the wise king listens to his queen and his wizard before making decisions
"""
# Train Bigram and Trigram models
bigram=NgramLanguageModel(n=2)
trigram=NgramLanguageModel(n=3)
bigram.train(text)
trigram.train(text)

# --- Predictions ---
print("=== BIGRAM MODEL ===")
print("Context: ['the']")
for word, prob in bigram.predict_next(["the"]):
    print(f"  P('{word}' | 'the') = {prob:.4f}")

print("\n=== TRIGRAM MODEL ===")
print("Context: ['the', 'king']")
for word, prob in trigram.predict_next(["the", "king"]):
    print(f"  P('{word}' | 'the king') = {prob:.4f}")

# --- Text generation ---
print("\n=== TEXT GENERATION ===")
print("Bigram  seed='the king':")
print(" ", bigram.generate("the king", max_words=15))

print("\nTrigram seed='the king':")
print(" ", trigram.generate("the king", max_words=15))

# --- Perplexity comparison ---
test_sentence = "the king rules the kingdom"
weird_sentence = "wizard the kingdom rules ancient"
print(f"\n=== PERPLEXITY ===")
print(f"Normal sentence:  {trigram.perplexity(test_sentence):.2f}")
print(f"Weird sentence:   {trigram.perplexity(weird_sentence):.2f}")
print("(Lower perplexity = model finds text more natural)")


=== BIGRAM MODEL ===
Context: ['the']
  P('kingdom' | 'the') = 0.0909
  P('king' | 'the') = 0.0727
  P('princess' | 'the') = 0.0364
  P('wizard' | 'the') = 0.0364
  P('knight' | 'the') = 0.0364

=== TRIGRAM MODEL ===
Context: ['the', 'king']
  P('the' | 'the king') = 0.0455
  P('rules' | 'the king') = 0.0455
  P('and' | 'the king') = 0.0455
  P('many' | 'the king') = 0.0227
  P('king' | 'the king') = 0.0227

=== TEXT GENERATION ===
Bigram  seed='the king':
  None

Trigram seed='the king':
  None

=== PERPLEXITY ===
Normal sentence:  21.33
Weird sentence:   16509636.22
(Lower perplexity = model finds text more natural)
